# Medicine Identification AI Model Training 🏥💊

This notebook demonstrates the complete pipeline for training a deep learning model to identify medicines from images using PyTorch and MobileNetV2 architecture.

## 📋 Table of Contents
1. [Dataset Preparation](#dataset-preparation)
2. [Data Preprocessing](#data-preprocessing)
3. [Model Architecture](#model-architecture)
4. [Training Configuration](#training-configuration)
5. [Training Process](#training-process)
6. [Model Evaluation](#model-evaluation)
7. [Model Saving](#model-saving)
8. [Inference Testing](#inference-testing)


## 1. Dataset Preparation 📊

### Dataset Structure
```
Mobile-Captured Pharmaceutical Medication Packages/
├── GTN 50 ml cream/
│   ├── 1.jpg
│   ├── 2.jpg
│   └── ...
├── Paracetamol 500mg/
│   ├── 1.jpg
│   ├── 2.jpg
│   └── ...
└── ... (150+ medicine classes)
```

### Dataset Statistics
- **Total Images**: 15,000+
- **Medicine Classes**: 150+
- **Images per Class**: 50-200
- **Image Resolution**: Various (resized to 224x224)
- **Format**: JPG, PNG


In [ ]:
# Import necessary libraries
import os
import shutil
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import time
import copy
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


In [ ]:
# Dataset preparation and train/validation split
nested_dir = "/kaggle/input/mobile-captured-pharmaceutical-medication-packages/Mobile-Captured Pharmaceutical Medication Packages"
output_dir = "/kaggle/working/medicine_dataset"
print("✅ Using dataset from:", nested_dir)

train_ratio = 0.8
train_dir = os.path.join(output_dir, "train")
val_dir = os.path.join(output_dir, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

packs = [p for p in os.listdir(nested_dir) if os.path.isdir(os.path.join(nested_dir, p))]

for pack in tqdm(packs):
    pack_path = os.path.join(nested_dir, pack)
    images = [img for img in os.listdir(pack_path) if img.lower().endswith(('.jpg','.jpeg','.png','.JPG'))]
    if len(images) == 0:
        continue
    random.shuffle(images)
    split_idx = int(len(images) * train_ratio)
    train_imgs, val_imgs = images[:split_idx], images[split_idx:]

    os.makedirs(os.path.join(train_dir, pack), exist_ok=True)
    os.makedirs(os.path.join(val_dir, pack), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(pack_path, img), os.path.join(train_dir, pack, img))
    for img in val_imgs:
        shutil.copy(os.path.join(pack_path, img), os.path.join(val_dir, pack, img))

print(f"✅ Dataset split complete. {len(packs)} classes processed.")


In [ ]:
# Data transforms and loaders
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

# Data loaders
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=2)
               for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print("✅ Number of classes:", len(class_names))
print("📦 Sample classes:", class_names[:5])


In [ ]:
# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize MobileNetV2
model = models.mobilenet_v2(pretrained=True)
for param in model.features.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)


In [ ]:
# Training function
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0
            for inputs, labels in tqdm(dataloaders[phase]):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            if phase == 'train': scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
    print(f"\n🏁 Best Validation Accuracy: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

# Start training
model = train_model(model, criterion, optimizer, scheduler, num_epochs=10)


In [ ]:
# Save the trained model
torch.save(model.state_dict(), "/kaggle/working/medicine_model.pth")
print("💾 Model saved successfully at /kaggle/working/medicine_model.pth")

# Test prediction function
from PIL import Image

def predict_image(image_path):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    transform = data_transforms['val']
    img_t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_t)
        _, preds = torch.max(outputs, 1)
    print("🧠 Predicted Medicine:", class_names[preds[0]])

# Test on a sample image
predict_image("/kaggle/working/medicine_dataset/val/GTN 50 ml cream/1.jpg")


In [1]:
pip install torch torchvision torchaudio opencv-python tqdm

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aryashah2k/mobile-captured-pharmaceutical-medication-packages")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'mobile-captured-pharmaceutical-medication-packages' dataset.
Path to dataset files: /kaggle/input/mobile-captured-pharmaceutical-medication-packages


In [13]:
import os

path = kagglehub.dataset_download("aryashah2k/mobile-captured-pharmaceutical-medication-packages")
print("Root path:", path)

# Inspect the first few levels
for root, dirs, files in os.walk(path):
    print("📂", root)
    if dirs:
        print("   ├── Subfolders:", dirs[:5])
    if files:
        print("   ├── Files:", files[:5])
    print()
    break  # only show the first level


Using Colab cache for faster access to the 'mobile-captured-pharmaceutical-medication-packages' dataset.
Root path: /kaggle/input/mobile-captured-pharmaceutical-medication-packages
📂 /kaggle/input/mobile-captured-pharmaceutical-medication-packages
   ├── Subfolders: ['Mobile-Captured Pharmaceutical Medication Packages']



In [14]:
for root, dirs, files in os.walk(os.path.join(path, "Mobile-Captured Pharmaceutical Medication Packages")):
    print("📂", root)
    if dirs:
        print("   ├── Subfolders:", dirs[:5])
    if files:
        print("   ├── Files:", files[:5])
    print()
    break


📂 /kaggle/input/mobile-captured-pharmaceutical-medication-packages/Mobile-Captured Pharmaceutical Medication Packages
   ├── Subfolders: ['Cemicresto 28 tablets', 'GTN 50 ml cream', 'Vitamax 15 capsules', 'Vitacid C 12 tablets', 'Reparil-Gel N 40 g gel']
   ├── Files: ['drug list.xlsx']



In [15]:
nested_dir = "/kaggle/input/mobile-captured-pharmaceutical-medication-packages/Mobile-Captured Pharmaceutical Medication Packages"
output_dir = "/kaggle/working/medicine_dataset"
print("✅ Using dataset from:", nested_dir)


✅ Using dataset from: /kaggle/input/mobile-captured-pharmaceutical-medication-packages/Mobile-Captured Pharmaceutical Medication Packages


In [16]:
import os, shutil, random
from tqdm import tqdm

train_ratio = 0.8
train_dir = os.path.join(output_dir, "train")
val_dir = os.path.join(output_dir, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

packs = [p for p in os.listdir(nested_dir) if os.path.isdir(os.path.join(nested_dir, p))]

for pack in tqdm(packs):
    pack_path = os.path.join(nested_dir, pack)
    images = [img for img in os.listdir(pack_path) if img.lower().endswith(('.jpg','.jpeg','.png','.JPG'))]
    if len(images) == 0:
        continue
    random.shuffle(images)
    split_idx = int(len(images) * train_ratio)
    train_imgs, val_imgs = images[:split_idx], images[split_idx:]

    os.makedirs(os.path.join(train_dir, pack), exist_ok=True)
    os.makedirs(os.path.join(val_dir, pack), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(pack_path, img), os.path.join(train_dir, pack, img))
    for img in val_imgs:
        shutil.copy(os.path.join(pack_path, img), os.path.join(val_dir, pack, img))

print(f"✅ Dataset split complete. {len(packs)} classes processed.")


100%|██████████| 150/150 [01:19<00:00,  1.89it/s]

✅ Dataset split complete. 150 classes processed.


In [17]:
!pip install torch torchvision torchaudio tqdm


In [18]:
import torch, time, copy
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

data_dir = output_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- TRANSFORMS ---
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

# --- DATA LOADERS ---
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=2)
               for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print("✅ Number of classes:", len(class_names))
print("📦 Sample classes:", class_names[:5])

# --- MODEL SETUP ---
model = models.mobilenet_v2(pretrained=True)
for param in model.features.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss, running_corrects = 0.0, 0
            for inputs, labels in tqdm(dataloaders[phase]):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            if phase == 'train': scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
    print(f"\n🏁 Best Validation Accuracy: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

model = train_model(model, criterion, optimizer, scheduler, num_epochs=10)
torch.save(model.state_dict(), "/kaggle/working/medicine_model.pth")
print("💾 Model saved successfully at /kaggle/working/medicine_model.pth")


✅ Number of classes: 150
📦 Sample classes: ['Acretin 30 g cream', 'Adol 24 caplets', 'Aggrex 60 tablets', 'Airoplast nan Tape', 'All-Vent 125 ml syrup']
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 13.6M/13.6M [00:00<00:00, 104MB/s] 



Epoch 1/10


100%|██████████| 188/188 [05:14<00:00,  1.67s/it]


train Loss: 4.0791 Acc: 0.2210


...

val Loss: 0.2739 Acc: 0.9456

Epoch 10/10


100%|██████████| 188/188 [05:00<00:00,  1.60s/it]


train Loss: 0.2506 Acc: 0.9750


100%|██████████| 57/57 [01:27<00:00,  1.54s/it]

val Loss: 0.2765 Acc: 0.9400

🏁 Best Validation Accuracy: 0.9456
💾 Model saved successfully at /kaggle/working/medicine_model.pth


In [20]:
from PIL import Image

def predict_image(image_path):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    transform = data_transforms['val']
    img_t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_t)
        _, preds = torch.max(outputs, 1)
    print("🧠 Predicted Medicine:", class_names[preds[0]])

predict_image("/download (1).jpg")


🧠 Predicted Medicine: Adol 24 caplets


In [21]:
import torch
from torchvision import models, transforms
from PIL import Image
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the same transforms used during validation
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load class names (you'll need to have these saved or recreate them)
# If you don't have class_names saved, you can recreate from directory structure
data_dir = "/kaggle/working/medicine_dataset"
class_names = sorted([d for d in os.listdir(os.path.join(data_dir, "train"))
                     if os.path.isdir(os.path.join(data_dir, "train", d))])
print("✅ Loaded classes:", len(class_names))

# Initialize model architecture (same as training)
model = models.mobilenet_v2(pretrained=False)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = torch.nn.Linear(num_ftrs, len(class_names))

# Load trained weights
model_path = "/kaggle/working/medicine_model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()  # Set to evaluation mode

print("✅ Model loaded successfully!")

def predict_image(image_path, model, class_names, transform):
    """
    Predict the class of an image
    """
    # Load and preprocess image
    img = Image.open(image_path).convert('RGB')
    img_t = transform(img).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        outputs = model(img_t)
        _, predicted = torch.max(outputs, 1)
        confidence = torch.nn.functional.softmax(outputs, dim=1)[0] * 100

    predicted_class = class_names[predicted[0]]
    confidence_score = confidence[predicted[0]].item()

    print(f"🧠 Predicted Medicine: {predicted_class}")
    print(f"📊 Confidence: {confidence_score:.2f}%")

    # Show top 3 predictions
    top3_conf, top3_idx = torch.topk(confidence, 3)
    print("\n🏆 Top 3 Predictions:")
    for i, (idx, conf) in enumerate(zip(top3_idx, top3_conf)):
        print(f"  {i+1}. {class_names[idx]} - {conf:.2f}%")

    return predicted_class, confidence_score

# Test the prediction
test_image_path = "/download (1).jpg"
if os.path.exists(test_image_path):
    predict_image(test_image_path, model, class_names, data_transforms)
else:
    print("❌ Test image not found, trying to find any test image...")
    # Find any image in validation set for testing
    for root, dirs, files in os.walk("/kaggle/working/medicine_dataset/val"):
        if files:
            test_image_path = os.path.join(root, files[0])
            print(f"🔍 Testing with: {test_image_path}")
            predict_image(test_image_path, model, class_names, data_transforms)
            break

✅ Loaded classes: 150
✅ Model loaded successfully!


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


🧠 Predicted Medicine: Adol 24 caplets
📊 Confidence: 36.26%

🏆 Top 3 Predictions:
  1. Adol 24 caplets - 36.26%
  2. Dalacin C 10 capsules - 10.83%
  3. B.B.C. 25 ml spray solution - 7.79%
